<center>

$\Huge \textbf{Universidad Nacional Autónoma de México}$  
$\Huge \textbf{Facultad de Ciencias}$  
<p align="center">
  <img src="https://www.icat.unam.mx/wp-content/uploads/2021/11/Copia-de-LogoUNAM.-Azul.-Fondo-transparente.png" alt="UNAM" width="200"/>
</p>

<hr style="height:3px; background-color:#0B6E4F; border:none;"/>


$\LARGE \textbf{Inteligencia Artificial}$  

$\Large \textit{Laboratorio 2.3}$  


\begin{array}{rl}
\textbf{Docente:} & Dra. Jessica Sarahi Méndez Rincón \\[6pt]
\textbf{Ayudante de laboratorio:} & Diego Eduardo Peña Villegas \\[6pt]
\textbf{Alumna:} & Santana de la Rosa \\[6pt]
\textbf{Fecha de realización:} & 25/02/2026
\end{array}

</center>

---
# 1. Distancia de Levenshtein

## 1.1 Fundamentos Matemáticos

La **distancia de Levenshtein** (o distancia de edición) entre dos cadenas $a$ y $b$ de longitudes $|a| = n$ y $|b| = m$ se define recursivamente:

$$
\text{lev}(a, b) =
\begin{cases}
|a| & \text{si } |b| = 0 \\
|b| & \text{si } |a| = 0 \\
\text{lev}(\text{tail}(a),\, \text{tail}(b)) & \text{si } a[0] = b[0] \\
1 + \min\!\begin{cases}
  \text{lev}(\text{tail}(a),\, b) & \text{(eliminación)} \\
  \text{lev}(a,\, \text{tail}(b)) & \text{(inserción)} \\
  \text{lev}(\text{tail}(a),\, \text{tail}(b)) & \text{(sustitución)}
\end{cases} & \text{en otro caso}
\end{cases}
$$

donde $\text{tail}(s)$ denota la cadena $s$ sin su primer carácter.

Para computarlo de forma eficiente usamos **programación dinámica** con una matriz $D \in \mathbb{Z}^{(n+1)\times(m+1)}$:

$$D[i][j] = \min\bigl(D[i-1][j]+1,\; D[i][j-1]+1,\; D[i-1][j-1] + c_{ij}\bigr)$$

con $c_{ij} = 0$ si $a[i-1] = b[j-1]$, y $c_{ij} = 1$ en caso contrario.

**Complejidad:** $O(n \cdot m)$ tiempo y espacio (optimizable a $O(\min(n,m))$ en espacio).

## 1.2 Descripción Formal del Algoritmo

El algoritmo opera en tres fases:
1. **Inicialización** — la primera fila y columna de la matriz representan el costo de transformar cualquier cadena en la cadena vacía: $D[i][0] = i$ y $D[0][j] = j$.
2. **Relleno** — para cada par $(i,j)$ se evalúan las tres operaciones y se toma el mínimo.
3. **Resultado** — $D[n][m]$ contiene la distancia mínima de edición entre $a$ y $b$.

## 1.3 Pseudocódigo

```
función levenshtein(a: cadena, b: cadena) → entero:
    n ← longitud(a)
    m ← longitud(b)

    // Crear matriz (n+1) × (m+1) inicializada en 0
    D ← matriz[n+1][m+1]

    // Casos base
    para i de 0 a n:  D[i][0] ← i
    para j de 0 a m:  D[0][j] ← j

    // Relleno dinámico
    para i de 1 a n:
        para j de 1 a m:
            si a[i-1] = b[j-1]:
                costo ← 0
            sino:
                costo ← 1
            D[i][j] ← mínimo(
                D[i-1][j]   + 1,      // eliminación
                D[i][j-1]   + 1,      // inserción
                D[i-1][j-1] + costo   // sustitución
            )

    retornar D[n][m]
```


In [ ]:
# No se requieren dependencias externas para Levenshtein
print("Levenshtein: listo para ejecutar")

## 1.4 Implementación con Buenas Prácticas

In [ ]:
from typing import List, Tuple


def levenshtein(a: str, b: str) -> int:
    """
    Calcula la distancia de Levenshtein entre dos cadenas.

    Args:
        a: Primera cadena.
        b: Segunda cadena.

    Returns:
        Número mínimo de operaciones (inserción, eliminación, sustitución)
        para transformar ``a`` en ``b``.

    Raises:
        TypeError: Si alguno de los argumentos no es una cadena de texto.

    Examples:
        >>> levenshtein("gato", "pato")
        1
        >>> levenshtein("", "abc")
        3
        >>> levenshtein("Messi", "Messi")
        0
    """
    if not isinstance(a, str) or not isinstance(b, str):
        raise TypeError(f"Se esperan cadenas; se recibió {type(a)} y {type(b)}")

    n, m = len(a), len(b)

    # ── Casos base rápidos (sin construir la matriz) ──────────────────────────
    if n == 0:
        return m
    if m == 0:
        return n

    # ── Construir la matriz D de dimensiones (n+1) × (m+1) ──────────────────
    # D[i][j] = distancia entre a[:i] y b[:j]
    D: List[List[int]] = [[0] * (m + 1) for _ in range(n + 1)]

    # Inicializar primera fila y columna (casos base)
    for i in range(n + 1):
        D[i][0] = i
    for j in range(m + 1):
        D[0][j] = j

    # ── Relleno por programación dinámica ────────────────────────────────────
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            costo = 0 if a[i - 1] == b[j - 1] else 1

            D[i][j] = min(
                D[i - 1][j] + 1,        # eliminación
                D[i][j - 1] + 1,        # inserción
                D[i - 1][j - 1] + costo # sustitución
            )

    return D[n][m]


def mostrar_matriz(a: str, b: str) -> None:
    """Imprime la matriz de distancias de edición de forma legible."""
    n, m = len(a), len(b)
    D: List[List[int]] = [[0] * (m + 1) for _ in range(n + 1)]

    for i in range(n + 1):
        D[i][0] = i
    for j in range(m + 1):
        D[0][j] = j

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            costo = 0 if a[i - 1] == b[j - 1] else 1
            D[i][j] = min(D[i-1][j] + 1, D[i][j-1] + 1, D[i-1][j-1] + costo)

    # Encabezado
    encabezado = f"{'':>6}" + "".join(f"{c:>5}" for c in " " + b)
    print(encabezado)
    print("─" * len(encabezado))

    filas = [" "] + list(a)
    for i, fila in enumerate(D):
        print(f"{filas[i]:>5} |" + "".join(f"{v:>5}" for v in fila))


# ── Pruebas unitarias simples ─────────────────────────────────────────────────
casos: List[Tuple[str, str, int]] = [
    ("Messi", "Messi", 0),       # cadenas idénticas
    ("Gol",   "Gool",  1),       # inserción
    ("gato",  "pato",  1),       # sustitución
    ("",      "futbol", 6),      # cadena vacía
    ("kitten", "sitting", 3),   # ejemplo clásico
]

print("═" * 55)
print(f"{'Cadena A':>10}  {'Cadena B':>10}  {'Esperado':>8}  {'Obtenido':>8}  {'OK':>4}")
print("═" * 55)

todos_ok = True
for (ca, cb, esperado) in casos:
    resultado = levenshtein(ca, cb)
    ok = resultado == esperado
    todos_ok = todos_ok and ok
    emoji = "✅" if ok else "❌"
    print(f"{ca:>10}  {cb:>10}  {esperado:>8}  {resultado:>8}  {emoji}")

print("═" * 55)
print("Todas las pruebas pasaron" if todos_ok else "FALLO en alguna prueba")


### Visualización de la Matriz de Edición

In [ ]:
print("Matriz de distancias de edición: 'kitten' → 'sitting'\n")
mostrar_matriz("kitten", "sitting")
print()
print(f"Distancia de Levenshtein: {levenshtein('kitten', 'sitting')}")


### Aplicación: Corrector ortográfico de términos futbolísticos

In [ ]:
VOCABULARIO_FUTBOL = [
    "gol", "portero", "delantero", "defensa", "arbitro",
    "penalti", "fuera de juego", "saque", "tarjeta", "balon"
]


def corregir_termino(termino: str, vocabulario: List[str], umbral: int = 3) -> str:
    """
    Retorna la palabra del vocabulario más cercana al término dado,
    siempre que su distancia no supere ``umbral``.

    Args:
        termino:     Término escrito por el usuario.
        vocabulario: Lista de palabras correctas.
        umbral:      Distancia máxima aceptable.

    Returns:
        La corrección sugerida o un mensaje indicando que no hay coincidencia.
    """
    termino = termino.lower().strip()
    mejor_palabra, mejor_distancia = "", float("inf")

    for palabra in vocabulario:
        d = levenshtein(termino, palabra)
        if d < mejor_distancia:
            mejor_distancia, mejor_palabra = d, palabra

    if mejor_distancia <= umbral:
        return f"'{termino}' → '{mejor_palabra}' (distancia = {mejor_distancia})"
    return f"'{termino}' → sin coincidencia cercana (distancia mínima = {mejor_distancia})"


errores_tipicos = ["goool", "portro", "arvitroo", "penaltte", "valón"]

print("Corrector ortográfico futbolístico")
print("─" * 45)
for err in errores_tipicos:
    print(" ", corregir_termino(err, VOCABULARIO_FUTBOL))


---
# 2. Word2Vec (Skip-Gram con Descenso de Gradiente)

## 2.1 Fundamentos Matemáticos

**Word2Vec** aprende representaciones vectoriales densas de palabras
(embeddings) de manera que palabras semánticamente similares queden
cerca en el espacio vectorial.

### Arquitectura Skip-Gram

Dado el vocabulario $\mathcal{V}$ de tamaño $V$ y una dimensión de
embedding $d$, se definen dos matrices:

$$W_1 \in \mathbb{R}^{V \times d} \quad \text{(embeddings de entrada)}$$
$$W_2 \in \mathbb{R}^{d \times V} \quad \text{(embeddings de salida)}$$

Para la palabra central $w_c$ (representada como vector one-hot
$\mathbf{x} \in \{0,1\}^V$):

**Paso hacia adelante (forward):**

$$\mathbf{h} = W_1^\top \mathbf{x} \in \mathbb{R}^d \quad \text{(capa oculta)}$$
$$\mathbf{u} = W_2^\top \mathbf{h} \in \mathbb{R}^V$$
$$\hat{\mathbf{y}} = \text{softmax}(\mathbf{u}), \quad \hat{y}_j = \frac{e^{u_j}}{\sum_{k=1}^V e^{u_k}}$$

**Función de pérdida (cross-entropy):**

$$\mathcal{L} = -\sum_{c \in \text{contexto}} \log \hat{y}_{c}$$

**Descenso de gradiente (backpropagation):**

$$\mathbf{e} = \hat{\mathbf{y}} - \mathbf{y} \quad \text{(error)}$$
$$\frac{\partial \mathcal{L}}{\partial W_2} = \mathbf{h} \otimes \mathbf{e}$$
$$\frac{\partial \mathcal{L}}{\partial W_1[w_c]} = W_2 \, \mathbf{e}$$

Actualización con tasa de aprendizaje $\eta$:

$$W_2 \leftarrow W_2 - \eta \, \mathbf{h} \otimes \mathbf{e}$$
$$W_1[w_c] \leftarrow W_1[w_c] - \eta \, (W_2 \, \mathbf{e})$$

La **similitud coseno** entre dos vectores se usa para medir cercanía semántica:

$$\cos(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\| \|\mathbf{v}\|}$$

## 2.2 Descripción del Algoritmo

El entrenamiento sigue estos pasos:
1. **Corpus** — colección de oraciones tokenizadas.
2. **Vocabulario** — mapeo `palabra → índice` a partir del corpus.
3. **Inicialización** — pesos aleatorios pequeños para $W_1$ y $W_2$.
4. **Ventana de contexto** — para cada palabra central, los vecinos dentro de la ventana se usan como pares de entrenamiento.
5. **Forward + Backward** — calcular $\hat{\mathbf{y}}$, el error $\mathbf{e}$ y actualizar los pesos.
6. **Embeddings finales** — las filas de $W_1$ son los vectores de palabras.

## 2.3 Pseudocódigo

```
función word2vec(corpus, dim_emb, ventana, épocas, η):
    vocab  ← construir_vocabulario(corpus)
    V      ← |vocab|
    W1     ← inicializar(V × dim_emb)   // embeddings de entrada
    W2     ← inicializar(dim_emb × V)   // embeddings de salida

    para cada época de 1 a épocas:
        para cada oración en corpus:
            para cada posición i:
                w_c ← oración[i]
                para c en rango(max(0, i-ventana), min(|oración|, i+ventana+1)):
                    si c ≠ i:
                        w_ctx ← oración[c]
                        x     ← one_hot(w_c, V)
                        y_real← one_hot(w_ctx, V)

                        // Forward
                        h     ← W1[índice(w_c)]
                        u     ← W2ᵀ · h
                        ŷ     ← softmax(u)

                        // Error
                        e     ← ŷ − y_real

                        // Backward
                        W2    ← W2 − η · (h ⊗ e)
                        W1[índice(w_c)] ← W1[índice(w_c)] − η · (W2 · e)

    retornar W1   // filas = vectores de palabras
```


## 2.4 Implementación con Buenas Prácticas

In [ ]:
import numpy as np
from typing import Dict, List, Optional


# ─── Utilidades ───────────────────────────────────────────────────────────────

def softmax(x: np.ndarray) -> np.ndarray:
    """Softmax numéricamente estable (resta el máximo para evitar overflow)."""
    x_estable = x - np.max(x)
    exp_x = np.exp(x_estable)
    return exp_x / exp_x.sum()


def similitud_coseno(v1: np.ndarray, v2: np.ndarray) -> float:
    """Similitud coseno entre dos vectores. Retorna valor en [-1, 1]."""
    norma_v1 = np.linalg.norm(v1)
    norma_v2 = np.linalg.norm(v2)
    if norma_v1 == 0 or norma_v2 == 0:
        return 0.0
    return float(np.dot(v1, v2) / (norma_v1 * norma_v2))


# ─── Clase principal ──────────────────────────────────────────────────────────

class Word2Vec:
    """
    Implementación desde cero del modelo Skip-Gram de Word2Vec.

    Args:
        dim_embedding: Dimensión del espacio de embeddings.
        ventana:       Tamaño de la ventana de contexto.
        tasa_aprendizaje: Hiperparámetro η del descenso de gradiente.
        epocas:        Número de épocas de entrenamiento.
        semilla:       Semilla aleatoria para reproducibilidad.
    """

    def __init__(
        self,
        dim_embedding: int = 10,
        ventana: int = 2,
        tasa_aprendizaje: float = 0.01,
        epocas: int = 100,
        semilla: Optional[int] = 42,
    ) -> None:
        self.d = dim_embedding
        self.ventana = ventana
        self.eta = tasa_aprendizaje
        self.epocas = epocas
        self.semilla = semilla

        # Se llenan durante el entrenamiento
        self.vocab: Dict[str, int] = {}
        self.vocab_inv: Dict[int, str] = {}
        self.V: int = 0
        self.W1: np.ndarray = np.array([])
        self.W2: np.ndarray = np.array([])

    # ── Construcción del vocabulario ─────────────────────────────────────────

    def _construir_vocabulario(self, corpus: List[List[str]]) -> None:
        """Asigna un índice único a cada palabra del corpus."""
        palabras_unicas = sorted({p for oracion in corpus for p in oracion})
        self.vocab = {p: i for i, p in enumerate(palabras_unicas)}
        self.vocab_inv = {i: p for p, i in self.vocab.items()}
        self.V = len(self.vocab)

    # ── Inicialización de pesos ──────────────────────────────────────────────

    def _inicializar_pesos(self) -> None:
        """Inicialización Xavier/Glorot para acelerar la convergencia."""
        if self.semilla is not None:
            np.random.seed(self.semilla)
        escala = np.sqrt(2.0 / (self.V + self.d))
        self.W1 = np.random.randn(self.V, self.d) * escala
        self.W2 = np.random.randn(self.d, self.V) * escala

    # ── Forward pass ─────────────────────────────────────────────────────────

    def _forward(self, idx_central: int):
        """
        Propagación hacia adelante.

        Returns:
            Tupla (ŷ, h, u): distribución de probabilidad, vector oculto y logits.
        """
        h = self.W1[idx_central]            # extracción directa (one-hot × W1)
        u = self.W2.T @ h                   # proyección a espacio del vocabulario
        y_hat = softmax(u)
        return y_hat, h, u

    # ── Backward pass ────────────────────────────────────────────────────────

    def _backward(
        self,
        error: np.ndarray,
        h: np.ndarray,
        idx_central: int,
    ) -> None:
        """Actualización de pesos mediante descenso de gradiente estocástico."""
        dW2 = np.outer(h, error)             # gradiente de W2
        dW1 = self.W2 @ error               # gradiente de la fila de W1

        self.W2 -= self.eta * dW2
        self.W1[idx_central] -= self.eta * dW1

    # ── Entrenamiento ────────────────────────────────────────────────────────

    def entrenar(self, corpus: List[List[str]], verbose: bool = True) -> None:
        """
        Entrena el modelo Word2Vec con el corpus proporcionado.

        Args:
            corpus:  Lista de oraciones; cada oración es una lista de tokens.
            verbose: Si True, imprime la pérdida cada 10 épocas.
        """
        self._construir_vocabulario(corpus)
        self._inicializar_pesos()

        for epoca in range(1, self.epocas + 1):
            perdida_total = 0.0

            for oracion in corpus:
                for i, palabra_central in enumerate(oracion):
                    idx_c = self.vocab[palabra_central]

                    # Obtener palabras de contexto dentro de la ventana
                    inicio = max(0, i - self.ventana)
                    fin = min(len(oracion), i + self.ventana + 1)

                    for j in range(inicio, fin):
                        if j == i:
                            continue
                        idx_ctx = self.vocab[oracion[j]]

                        # Forward
                        y_hat, h, _ = self._forward(idx_c)

                        # Error: ŷ − y_real (one-hot en idx_ctx)
                        error = y_hat.copy()
                        error[idx_ctx] -= 1.0

                        # Pérdida cross-entropy
                        perdida_total -= np.log(y_hat[idx_ctx] + 1e-10)

                        # Backward
                        self._backward(error, h, idx_c)

            if verbose and (epoca % 10 == 0 or epoca == 1):
                print(f"  Época {epoca:>4}/{self.epocas}  |  Pérdida = {perdida_total:.4f}")

        if verbose:
            print("\n  Entrenamiento completado.")

    # ── Consultas semánticas ─────────────────────────────────────────────────

    def vector(self, palabra: str) -> np.ndarray:
        """Retorna el embedding de una palabra."""
        if palabra not in self.vocab:
            raise ValueError(f"'{palabra}' no está en el vocabulario.")
        return self.W1[self.vocab[palabra]]

    def similitud(self, p1: str, p2: str) -> float:
        """Similitud coseno entre dos palabras."""
        return similitud_coseno(self.vector(p1), self.vector(p2))

    def palabras_similares(self, palabra: str, top_n: int = 5) -> List[tuple]:
        """Retorna las ``top_n`` palabras más similares a ``palabra``."""
        if palabra not in self.vocab:
            raise ValueError(f"'{palabra}' no está en el vocabulario.")
        v_ref = self.vector(palabra)
        scores = [
            (other, similitud_coseno(v_ref, self.W1[idx]))
            for other, idx in self.vocab.items()
            if other != palabra
        ]
        return sorted(scores, key=lambda t: t[1], reverse=True)[:top_n]


print("Clase Word2Vec lista.")


### Entrenamiento y consultas semánticas

In [ ]:
# ── Corpus futbolístico ───────────────────────────────────────────────────────
corpus_futbol = [
    ["el", "delantero", "marcó", "el", "gol"],
    ["el", "atacante", "anotó", "el", "gol"],
    ["el", "portero", "detuvo", "el", "balón"],
    ["el", "arquero", "atajó", "la", "pelota"],
    ["el", "defensa", "bloqueó", "el", "tiro"],
    ["el", "mediocampista", "pasó", "el", "balón"],
    ["gol", "de", "penal", "marcó", "el", "delantero"],
    ["el", "equipo", "ganó", "el", "partido"],
    ["el", "árbitro", "pitó", "el", "penal"],
    ["la", "pelota", "entró", "al", "arco"],
]

# ── Entrenamiento ──────────────────────────────────────────────────────────────
print("Entrenando Word2Vec (Skip-Gram)...\n")
modelo_w2v = Word2Vec(
    dim_embedding=8,
    ventana=2,
    tasa_aprendizaje=0.05,
    epocas=200,
    semilla=42,
)
modelo_w2v.entrenar(corpus_futbol, verbose=True)


In [ ]:
# ── Similitudes semánticas ────────────────────────────────────────────────────
pares = [
    ("delantero", "atacante"),
    ("portero",   "arquero"),
    ("balón",     "pelota"),
    ("gol",       "penal"),
]

print("Similitudes coseno entre pares de palabras")
print("─" * 48)
for p1, p2 in pares:
    sim = modelo_w2v.similitud(p1, p2)
    barra = "█" * int(sim * 20) if sim > 0 else ""
    print(f"  {p1:>14} ↔ {p2:<14}  {sim:+.4f}  {barra}")

print()
print("Palabras más similares a 'gol':")
for (palabra, sim) in modelo_w2v.palabras_similares("gol", top_n=5):
    print(f"  {palabra:>15}  {sim:+.4f}")


---
# 3. Operador de Sobel (Detección de Bordes)

## 3.1 Fundamentos Matemáticos

El **operador de Sobel** estima el gradiente de la intensidad de una
imagen en escala de grises, resaltando los cambios abruptos que
corresponden a bordes.

Dada una imagen $I(x,y)$, el operador aplica dos **kernels de convolución** 3×3:

$$G_x = \begin{bmatrix}-1 & 0 & 1 \\ -2 & 0 & 2 \\ -1 & 0 & 1\end{bmatrix}$$

$$G_y = \begin{bmatrix}1 & 2 & 1 \\ 0 & 0 & 0 \\ -1 & -2 & -1\end{bmatrix}$$

La **respuesta en cada píxel** $(i,j)$ se calcula mediante la convolución discreta:

$$g_x(i,j) = \sum_{m=-1}^{1}\sum_{n=-1}^{1} I(i+m,\, j+n)\cdot G_x(m+1,\, n+1)$$
$$g_y(i,j) = \sum_{m=-1}^{1}\sum_{n=-1}^{1} I(i+m,\, j+n)\cdot G_y(m+1,\, n+1)$$

La **magnitud del gradiente** (mapa de bordes):

$$G(i,j) = \sqrt{g_x(i,j)^2 + g_y(i,j)^2}$$

La **dirección del gradiente** (normal al borde):

$$\theta(i,j) = \arctan\!\left(\frac{g_y(i,j)}{g_x(i,j)}\right)$$

### ¿Por qué funciona?

$G_x$ detecta cambios horizontales (bordes verticales) y $G_y$ detecta
cambios verticales (bordes horizontales). La combinación pitagórica
produce una detección isotrópica de bordes en todas las direcciones.

## 3.2 Descripción del Algoritmo

1. **Preprocesamiento** — convertir la imagen a escala de grises y, si
   es de tipo flotante, normalizar al rango $[0, 255]$.
2. **Padding** — agregar una franja de ceros alrededor para preservar
   las dimensiones y evitar artefactos en los bordes.
3. **Convolución** — deslizar los kernels $G_x$ y $G_y$ sobre cada
   ventana 3×3 de la imagen con padding.
4. **Magnitud** — combinar $g_x$ y $g_y$ pitagóricamente.
5. **Normalización** — escalar el resultado a $[0, 255]$ para
   visualización como imagen de 8 bits.

## 3.3 Pseudocódigo

```
función sobel(imagen: matriz[H×W]) → matriz[H×W]:
    Gx ← [[-1,0,1],[-2,0,2],[-1,0,1]]
    Gy ← [[1,2,1],[0,0,0],[-1,-2,-1]]

    salida ← matriz zeros[H×W]
    img_pad ← agregar_padding(imagen, 1)   // relleno con ceros

    para i de 0 a H-1:
        para j de 0 a W-1:
            region ← img_pad[i:i+3, j:j+3]
            gx ← suma(region ⊙ Gx)        // producto de Hadamard + suma
            gy ← suma(region ⊙ Gy)
            salida[i][j] ← sqrt(gx² + gy²)

    // Normalizar a [0, 255]
    salida ← salida × 255 / máximo(salida)
    retornar salida como uint8
```


## 3.4 Implementación con Buenas Prácticas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple


# ─── Kernels de Sobel (constantes del operador) ────────────────────────────────
KERNEL_GX: np.ndarray = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1],
], dtype=np.float64)

KERNEL_GY: np.ndarray = np.array([
    [ 1,  2,  1],
    [ 0,  0,  0],
    [-1, -2, -1],
], dtype=np.float64)


# ─── Funciones auxiliares ─────────────────────────────────────────────────────

def normalizar_imagen(imagen: np.ndarray) -> np.ndarray:
    """
    Convierte cualquier imagen numérica a float en [0, 255].

    - Si el rango es [0.0, 1.0] (float), escala × 255.
    - Si ya está en [0, 255], simplemente convierte a float.
    """
    img = imagen.astype(np.float64)
    if img.max() <= 1.0:
        img = img * 255.0
    return img


def aplicar_kernel(imagen_pad: np.ndarray, kernel: np.ndarray,
                   i: int, j: int) -> float:
    """Aplica un kernel 3×3 centrado en el píxel (i, j) de la imagen con padding."""
    region = imagen_pad[i : i + 3, j : j + 3]
    return float(np.sum(region * kernel))


# ─── Algoritmo principal ──────────────────────────────────────────────────────

def sobel(
    imagen: np.ndarray,
    normalizar_salida: bool = True,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Aplica el operador de Sobel para detectar bordes en una imagen en
    escala de grises.

    Args:
        imagen:            Imagen 2D (H × W). Puede ser float [0,1] o uint8 [0,255].
        normalizar_salida: Si True, escala el resultado a [0, 255].

    Returns:
        Tupla (magnitud, gx, gy):
            - magnitud: mapa de bordes normalizado (uint8).
            - gx:       gradiente horizontal (float64).
            - gy:       gradiente vertical (float64).

    Raises:
        ValueError: Si la imagen no es 2D.
    """
    if imagen.ndim != 2:
        raise ValueError(
            f"Se espera imagen 2D (escala de grises); se recibió shape {imagen.shape}. "
            "Convierta a escala de grises antes de llamar a esta función."
        )

    img = normalizar_imagen(imagen)
    H, W = img.shape

    # Padding con ceros (1 fila/columna alrededor)
    img_pad = np.pad(img, pad_width=1, mode="constant", constant_values=0)

    gx = np.zeros((H, W), dtype=np.float64)
    gy = np.zeros((H, W), dtype=np.float64)

    # Convolución 2-D
    for i in range(H):
        for j in range(W):
            gx[i, j] = aplicar_kernel(img_pad, KERNEL_GX, i, j)
            gy[i, j] = aplicar_kernel(img_pad, KERNEL_GY, i, j)

    magnitud = np.hypot(gx, gy)          # sqrt(gx² + gy²), más estable

    if normalizar_salida and magnitud.max() > 0:
        magnitud = (magnitud / magnitud.max()) * 255.0

    return magnitud.astype(np.uint8), gx, gy


# ─── Función de visualización ─────────────────────────────────────────────────

def visualizar_sobel(
    original: np.ndarray,
    magnitud: np.ndarray,
    gx: np.ndarray,
    gy: np.ndarray,
    titulo: str = "Análisis de Bordes con Sobel",
) -> None:
    """Muestra la imagen original y los componentes del gradiente."""
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    fig.suptitle(titulo, fontsize=14, fontweight="bold")

    datos = [
        (original,  "Original (gris)",     "gray"),
        (magnitud,  "Magnitud |G|",         "gray"),
        (np.abs(gx),"Gradiente Gx",        "RdBu_r"),
        (np.abs(gy),"Gradiente Gy",        "RdBu_r"),
    ]

    for ax, (img, titulo_ax, cmap) in zip(axes, datos):
        ax.imshow(img, cmap=cmap)
        ax.set_title(titulo_ax, fontsize=11)
        ax.axis("off")

    plt.tight_layout()
    plt.show()


print("Operador de Sobel listo.")


### Prueba con imagen sintética

In [ ]:
# ── Imagen sintética con figuras geométricas ─────────────────────────────────
def crear_imagen_prueba(tamaño: int = 100) -> np.ndarray:
    """Genera una imagen en escala de grises con rectángulo y círculo."""
    img = np.zeros((tamaño, tamaño), dtype=np.float64)

    # Rectángulo (intensidad 200)
    img[20:50, 20:80] = 200.0

    # Círculo (intensidad 150)
    cy, cx, r = 70, 70, 20
    y_idx, x_idx = np.ogrid[:tamaño, :tamaño]
    mascara = (x_idx - cx)**2 + (y_idx - cy)**2 <= r**2
    img[mascara] = 150.0

    return img


img_prueba = crear_imagen_prueba()

# ── Aplicar Sobel ─────────────────────────────────────────────────────────────
magnitud, gx, gy = sobel(img_prueba)

# ── Verificar salida ──────────────────────────────────────────────────────────
print(f"Shape de entrada : {img_prueba.shape}")
print(f"Shape de salida  : {magnitud.shape}")
print(f"Valor máximo     : {magnitud.max()}")
print(f"Valor mínimo     : {magnitud.min()}")
print()

visualizar_sobel(img_prueba, magnitud, gx, gy,
                 titulo="Sobel — Imagen sintética (rectángulo + círculo)")


### Prueba con imagen real (astronauta de skimage)

In [ ]:
from skimage import data, color

# ── Cargar imagen real ────────────────────────────────────────────────────────
imagen_rgb = data.astronaut()
imagen_gris = color.rgb2gray(imagen_rgb)   # float [0, 1]

print(f"Imagen original  : {imagen_rgb.shape}  dtype={imagen_rgb.dtype}")
print(f"Escala de grises : {imagen_gris.shape}  dtype={imagen_gris.dtype}  ")
print(f"  rango: [{imagen_gris.min():.2f}, {imagen_gris.max():.2f}]")

# ── Aplicar Sobel ─────────────────────────────────────────────────────────────
# (la función normalizar_imagen detecta el rango [0,1] automáticamente)
magnitud_astro, gx_astro, gy_astro = sobel(imagen_gris)

visualizar_sobel(
    imagen_gris,
    magnitud_astro,
    gx_astro,
    gy_astro,
    titulo="Sobel — Astronauta (skimage)",
)


### Dirección del gradiente (orientación de bordes)

In [ ]:
# ── Dirección del gradiente ──────────────────────────────────────────────────
def calcular_orientacion(gx: np.ndarray, gy: np.ndarray) -> np.ndarray:
    """Retorna el ángulo del gradiente en grados [-180, 180]."""
    return np.degrees(np.arctan2(gy, gx))

orientacion = calcular_orientacion(gx_astro, gy_astro)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Magnitud y Orientación del Gradiente de Sobel", fontsize=13, fontweight="bold")

im0 = axes[0].imshow(magnitud_astro, cmap="gray")
axes[0].set_title("Magnitud |G|")
axes[0].axis("off")
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(orientacion, cmap="hsv")
axes[1].set_title("Orientación θ (°)")
axes[1].axis("off")
plt.colorbar(im1, ax=axes[1], fraction=0.046)

plt.tight_layout()
plt.show()

print("Orientación: rango [°]", f"[{orientacion.min():.1f}, {orientacion.max():.1f}]")


---
# 4. Conclusiones

| Algoritmo | Tipo | Complejidad | Aplicación principal |
|-----------|------|-------------|----------------------|
| **Levenshtein** | DP discreta | $O(n \cdot m)$ | Corrección ortográfica, búsqueda aproximada de cadenas |
| **Word2Vec** | Red neuronal shallow | $O(E \cdot |C| \cdot V \cdot d)$ | Semántica distribucional, NLP, sistemas de recomendación |
| **Sobel** | Convolución 2D | $O(H \cdot W)$ | Detección de bordes, visión computacional, preprocesamiento |

### Buenas prácticas aplicadas

- **Type hints** y **docstrings** completos (parámetros, retornos, excepciones, ejemplos).
- **Validación de entrada** antes de operar (tipos, dimensiones, rangos).
- **Separación de responsabilidades**: cada función hace una sola cosa.
- **Constantes nombradas** (`KERNEL_GX`, `KERNEL_GY`) en lugar de literales mágicos.
- **Estabilidad numérica**: softmax con resta del máximo, `np.hypot` en lugar de `sqrt`.
- **Reproducibilidad**: semilla aleatoria como parámetro del modelo.
- **Pruebas unitarias** para Levenshtein con casos conocidos.
